# Seasonality: what a "normal" year looks like at Lululime

**Why this matters.** The case asks whether the loyalty program (launched June 2024) and free repairs (launched April 2025) helped the business. Answering that means comparing periods. But a store's sales swing heavily through the year, so if a launch happens in a quiet month, a simple before/after comparison will mistake the season for a program effect. This notebook measures the seasonal pattern so those comparisons can be done fairly.

**Questions this notebook answers**
1. How much does revenue swing by calendar month, for the whole store and for each product category?
2. Do winter products (outerwear, hoodies) peak in Nov–Jan? When do summer products peak?
3. How big are the two holiday sales: Black Friday weekend and Boxing Week?

### Summary of the answers
- **December brings in ~1.9× an average month**, November 1.3×, September–October ~1.07×. **February, June and July are the quietest** (~0.76×). The pattern was the same before the competitor opened.
- **Outerwear and hoodies make ~half their yearly sales in Nov–Jan** (December ~2.7–2.9× an average month; June–July only ~0.2–0.3×). **Shorts and tank tops** are the mirror image, peaking May–August.
- **Black Friday + Boxing Week are 9 days and ~5% of yearly revenue**, costing ~$142K a year in discounts. A sale day sells ~1.4× an ordinary Nov–Dec day, so most of the December peak is full-price holiday shopping.
- **For anyone comparing periods:** loyalty launched in June and repairs in April, both quiet months. Repairs launched just as outerwear entered its slowest season.

## Data and definitions

**The data.** Every Lululime transaction from **1 Nov 2022 to 31 Oct 2026**, one row per item line, joined with product and customer details. The team cleaned it once in shared code (`src/`):
- `load("lines")` returns the cleaned, joined table: one row per item line.
- `C` holds values the team agreed on, such as `C.PURCHASE` = "Purchase" and `C.COMPETITOR_OPEN` = 1 Mar 2024 (the competitor's opening, inferred from the data in `a1_competitor_impact.ipynb`).

**Purchases only.** We measure when customers buy, so we exclude returns (processed weeks later) and repairs (free).

**Why the averages are fair.** The data covers exactly 4 years, so **every calendar month appears exactly 4 times** (e.g. four Decembers: 2022–2025). No month is over-represented.

**Terms**
| Term | Meaning |
|---|---|
| Seasonal index | Average revenue in a calendar month ÷ the average month × 100. **100 = a normal month**, 200 = double, 50 = half |
| Black Friday weekend | 25% off, 3 days in late November. Found in the data as items sold at exactly 75% of list price |
| Boxing Week | 30% off, 26–31 December. Found as items sold at exactly 70% of list price |
| Retail year | November to October, so each year contains one complete holiday season |

According to the case, these are the store's only discounts. The check below confirms no other discount levels appear.

In [ ]:
import sys
from pathlib import Path

# Make the repo's shared code (src/) importable; this notebook lives in analysis/a_competitor/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.colors import TwoSlopeNorm

from src.metrics import load      # shared loader for the cleaned tables
from src import config as C       # shared constants: dates, row types, paths

OUT = C.RESULTS_DIR / "a_competitor"     # charts are saved here
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold",
                     "axes.titlelocation": "left", "legend.frameon": False})
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

lines = load("lines")
p = lines[lines.row_type == C.PURCHASE].copy()       # purchases only
p["cal_month"] = p.date.dt.month                     # 1 = Jan ... 12 = Dec

# Flag the holiday sales from the discount actually given: 1 - price paid / list price
p["discount"] = (1 - p.unit_price_paid / p.price).round(2)
p["promo"] = p.discount.map({0.25: "Black Friday", 0.30: "Boxing Week"}).fillna("Full price")
assert set(p.discount.unique()) <= {0.0, 0.25, 0.30}, "unexpected discount level"

# Every calendar month should appear in exactly 4 different years
n_years = p.groupby("cal_month").date.apply(lambda d: d.dt.year.nunique())
assert (n_years == 4).all(), "each calendar month should appear 4 times"
print("Items sold by price type:")
print(p.promo.value_counts().to_string())

**What this shows.** Only three price levels exist: full price, 25% off (Black Friday) and 30% off (Boxing Week), matching the case. About 6% of items were sold on sale.

## 1. The store-wide seasonal pattern

In [ ]:
store = p.groupby("cal_month").revenue.sum() / 4                  # average revenue per calendar month
store_idx = (store / store.mean() * 100).rename(index=dict(enumerate(MONTHS, 1)))

# Check: the 4-year average includes the competitor's dip (Mar 2024 – Feb 2025).
# Compare it with the one full year before the competitor opened (Mar 2023 – Feb 2024).
pre = p[(p.date >= "2023-03-01") & (p.date < "2024-03-01")].groupby("cal_month").revenue.sum()
pre_idx = (pre / pre.mean() * 100).rename(index=dict(enumerate(MONTHS, 1)))
seasonal = pd.DataFrame({"All 4 years (Nov 2022 – Oct 2026)": store_idx,
                         "Pre-competitor year (Mar 2023 – Feb 2024)": pre_idx}).T
seasonal.columns.name = None
print(f"Correlation between the two profiles: {store_idx.corr(pre_idx):.2f}")
(seasonal.style.format("{:.0f}")
 .set_caption("<b>Store-wide seasonal index by calendar month</b> (100 = an average month). "
              "Example: Dec 194 = December brings in ~1.9× an average month. "
              "The second row is a check that the competitor years didn't distort the pattern.")
 .set_table_styles([{"selector": "caption", "props": "caption-side: top; text-align: left; font-size: 0.95em;"}]))

**What this shows.** December (194) is nearly double an average month, followed by November (127) and a back-to-school / autumn lift in September–October (~106). February, June and July (76–78) are the quietest. The pre-competitor year has **the same shape** (correlation 1.00): the competitor changed how much people bought, not when they bought.

### 1b. Does the way we calculate the index matter?
The simple index above assumes there's **no long-term trend**. Our data has one: a dip in 2024 after the competitor opened, then growth in 2025–26. It also ignores that months have different numbers of days. So we compare four methods:

| Method | How it works | Trade-off |
|---|---|---|
| **Average month** (used above) | Average revenue per calendar month ÷ average of all months | Simplest to explain. Ignores trend and month length |
| **Per open day** | Revenue per day the store was open (it's closed 25 December), then the same index | Fixes February's 28 days |
| **Ratio to moving average** | Each month ÷ the average of the 12 months around it (which removes the trend), then averaged by calendar month | Removes trend. Loses 6 months at each end of the data |
| **Regression** | A statistical model: log(monthly revenue) explained by month of year + a steady trend + a flag for the first competitor year (Mar 2024 – Feb 2025) | Separates season, trend and the competitor shock at once |

If all four agree, the seasonal pattern doesn't depend on the method.

In [ ]:
m = p.groupby(p.date.dt.to_period("M")).revenue.sum()          # 48 monthly totals
cal = m.index.month

# 1. average month (same as above)
avg = m.groupby(cal).mean()
idx_avg = avg / avg.mean() * 100

# 2. per open day (store closed on Dec 25)
open_days = pd.Series([per.days_in_month - (per.month == 12) for per in m.index], index=m.index)
pdm = (m / open_days).groupby(cal).mean()
idx_day = pdm / pdm.mean() * 100

# 3. ratio to a centred 12-month moving average (classical seasonal decomposition)
ma = m.rolling(12, center=True).mean()
ma = (ma + ma.shift(-1)) / 2            # average two 12-month windows so the result is centred on each month
ratio = (m / ma).dropna()
rma = ratio.groupby(ratio.index.month).mean()
idx_ma = rma / rma.mean() * 100

# 4. regression: log(revenue) ~ month dummies + trend + competitor-year flag (least squares, numpy only)
comp_year = (m.index >= C.COMPETITOR_OPEN.to_period("M")) & (m.index < (C.COMPETITOR_OPEN + pd.DateOffset(years=1)).to_period("M"))
X = pd.get_dummies(pd.Series(cal, name="m"), prefix="m", drop_first=True).astype(float)   # January is the baseline
X.insert(0, "const", 1.0)
X["trend"] = np.arange(len(m))
X["competitor_year"] = comp_year.astype(float)
beta, *_ = np.linalg.lstsq(X.values, np.log(m.values), rcond=None)
b = dict(zip(X.columns, beta))
month_eff = np.exp([0.0] + [b[f"m_{k}"] for k in range(2, 13)])
idx_reg = pd.Series(month_eff / month_eff.mean() * 100, index=range(1, 13))

methods = pd.DataFrame({
    "Average month (used)": idx_avg.values,
    "Per open day": idx_day.values,
    "Ratio to moving average": idx_ma.values,
    "Regression (trend + competitor)": idx_reg.values,
}, index=MONTHS).T
max_gap = (methods - methods.iloc[0]).abs().max().max()

print(f"Ratio-to-moving-average uses {len(ratio)} of {len(m)} months ({ratio.index.min()} to {ratio.index.max()})")
print(f"Largest difference from the simple index: {max_gap:.0f} points "
      f"(in {(methods - methods.iloc[0]).abs().max().idxmax()})")
first_year = m[comp_year].sum() / m[(m.index >= pd.Period("2023-03", "M")) & (m.index <= pd.Period("2024-02", "M"))].sum() - 1
print(f"Regression side result: competitor year = {np.exp(b['competitor_year']) - 1:+.0%} revenue, "
      f"after controlling for season and trend (simple comparison, Mar 2024–Feb 2025 vs the year before: {first_year:+.0%})")
print(f"Underlying trend: {np.exp(b['trend'] * 12) - 1:+.1%} per year")

(methods.style.format("{:.0f}")
 .set_caption("<b>Seasonal index by method</b> (100 = an average month). All four methods agree within "
              f"{max_gap:.0f} points, so the seasonal pattern is robust.")
 .set_table_styles([{"selector": "caption", "props": "caption-side: top; text-align: left; font-size: 0.95em;"}]))

**What this shows.**
- **All four methods agree within 6 points**, so the seasonal pattern doesn't depend on how it's calculated. The biggest gap is February (76 simple vs 82 per open day), partly because February has only 28 days.
- **Side result:** after removing season and trend, the regression estimates the **first competitor year cost ~14% of revenue** (a simple comparison with the year before gives −12%). The underlying trend is about +3% a year. This is an independent check on `a1_competitor_impact.ipynb`. It rests on only 48 monthly data points, so treat −14% as an estimate.

## 2. Seasonality by product category
Same index, calculated for each category separately (each row's 100 = that category's average month). Rows are ordered from winter products at the top to summer products at the bottom.

In [ ]:
cat = p.groupby(["category", "cal_month"]).revenue.sum().unstack() / 4
cat_idx = cat.div(cat.mean(axis=1), axis=0) * 100
order = ["Outerwear", "Hoodies & Sweaters", "Tops", "Leggings", "Accessories", "Sports Bras", "Tank Tops", "Shorts"]
cat_idx = cat_idx.loc[order]
heat = pd.concat([cat_idx, store_idx.set_axis(range(1, 13)).to_frame("Whole store").T])

fig, ax = plt.subplots(figsize=(11, 5))
im = ax.imshow(heat.values, cmap="RdBu_r", norm=TwoSlopeNorm(vcenter=100, vmin=25, vmax=300), aspect="auto")
ax.set_xticks(range(12), MONTHS)
ax.set_yticks(range(len(heat)), heat.index)
ax.axhline(len(heat) - 1.5, color="white", lw=3)        # separate the whole-store row
for i in range(heat.shape[0]):
    for j in range(12):
        v = heat.iat[i, j]
        ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8.5,
                color="white" if (v > 200 or v < 45) else "#222",
                fontweight="bold" if i == len(heat) - 1 else "normal")
ax.tick_params(length=0)
for s in ax.spines.values(): s.set_visible(False)
cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01)
cb.set_label("100 = an average month for that row", fontsize=8.5)
ax.set_title("December is ~2× a normal month; winter gear peaks at ~3× in Dec, summer gear in May–Aug", fontsize=11.5)
ax.set_xlabel("Seasonal index = average revenue in that calendar month ÷ the row's average month × 100 "
              "(Nov 2022 – Oct 2026, each month counted 4 times)", fontsize=8.5, color="#444")
plt.tight_layout(); fig.savefig(OUT / "a2_seasonality_heatmap.png", bbox_inches="tight"); plt.show()

In [ ]:
peaks = pd.DataFrame({
    "peak month": cat_idx.idxmax(axis=1).map(dict(enumerate(MONTHS, 1))),
    "peak index": cat_idx.max(axis=1).round(0),
    "low month": cat_idx.idxmin(axis=1).map(dict(enumerate(MONTHS, 1))),
    "low index": cat_idx.min(axis=1).round(0),
    "Nov–Jan share of year": cat[[11, 12, 1]].sum(axis=1) / cat.sum(axis=1),
})
(peaks.style.format({"Nov–Jan share of year": "{:.0%}", "peak index": "{:.0f}", "low index": "{:.0f}"})
 .set_caption("<b>Each category's busiest and quietest month</b>, and the share of its yearly revenue sold in Nov–Jan"))

**What this shows.** Three groups of products:
- **Winter products:** outerwear (Dec 272, Jun 31) and hoodies (Dec 293, Jun 23). Outerwear sells **49% of its year in Nov–Jan**.
- **Summer products:** shorts and tank tops (~170–185 in May–Aug, ~30 in Jan–Feb).
- **Year-round staples:** leggings, tops, sports bras and accessories, which are fairly flat apart from the December gift peak (~160–190).

**Why this matters for the repair program.** Free repairs launched on **1 April 2025**, just as outerwear entered its slowest months (April 60, June 31). A simple "before vs after launch" comparison of outerwear sales would show a large drop that is **pure seasonality**. Any test of whether repairs reduce jacket sales must compare the same months year over year.

## 3. How big are Black Friday weekend and Boxing Week?
Measured for each holiday season (Nov–Dec). "Discount given" = list price minus price paid on sale items: margin handed back to customers.

In [ ]:
# Retail year runs Nov–Oct, so each one contains exactly one holiday season
p["season_year"] = np.where(p.cal_month >= 11, p.date.dt.year, p.date.dt.year - 1)
rows = []
for y in sorted(p.season_year.unique()):
    yr = p[p.season_year == y]
    nd = yr[yr.cal_month.isin([11, 12])]                 # the holiday season
    days = nd.groupby("promo").date.nunique()
    rev = nd.groupby("promo").revenue.sum()
    disc = (nd.price * nd.quantity - nd.revenue).groupby(nd.promo).sum()
    rows.append({
        "retail year": f"Nov {y} – Oct {y + 1}",
        "BF days": days.get("Black Friday", 0), "BW days": days.get("Boxing Week", 0),
        "BF revenue": rev.get("Black Friday", 0), "BW revenue": rev.get("Boxing Week", 0),
        "promo share of year": (rev.get("Black Friday", 0) + rev.get("Boxing Week", 0)) / yr.revenue.sum(),
        "Nov–Dec share of year": nd.revenue.sum() / yr.revenue.sum(),
        "discount given": disc.get("Black Friday", 0) + disc.get("Boxing Week", 0),
        "promo $/day ÷ other Nov–Dec $/day": (rev.drop("Full price").sum() / days.drop("Full price").sum())
                                            / (rev["Full price"] / days["Full price"]),
    })
promo = pd.DataFrame(rows).set_index("retail year")
(promo.style.format({"BF revenue": "${:,.0f}", "BW revenue": "${:,.0f}", "discount given": "${:,.0f}",
                     "promo share of year": "{:.1%}", "Nov–Dec share of year": "{:.0%}",
                     "promo $/day ÷ other Nov–Dec $/day": "{:.2f}×"})
 .set_caption("<b>Holiday sales by retail year</b> (BF = Black Friday weekend, BW = Boxing Week)"))

In [ ]:
# Average revenue per day for four kinds of day
daily = p.groupby("date").agg(revenue=("revenue", "sum"),
                              promo=("promo", lambda s: s[s != "Full price"].iloc[0] if (s != "Full price").any() else "Full price"))
daily["bucket"] = np.select(
    [daily.promo == "Black Friday", daily.promo == "Boxing Week", daily.index.month.isin([11, 12])],
    ["Black Friday weekend", "Boxing Week", "Other Nov–Dec days"], default="Rest of year (Jan–Oct)")
per_day = daily.groupby("bucket").revenue.mean().reindex(
    ["Rest of year (Jan–Oct)", "Other Nov–Dec days", "Boxing Week", "Black Friday weekend"])

fig, ax = plt.subplots(figsize=(9, 3.6))
colors = ["#9AA5B1", "#5B8DB8", "#D1495B", "#D1495B"]
ax.barh(per_day.index, per_day.values, color=colors)
base = per_day.iloc[0]
for i, v in enumerate(per_day.values):
    ax.text(v + 400, i, f"${v/1e3:,.1f}K/day  ({v / base:.1f}× a normal day)", va="center", fontsize=9)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"${x/1e3:.0f}K"))
ax.set(title="Sale days sell ~2.5× a normal day, but only ~1.4× other holiday-season days",
       xlabel="Average purchase revenue per day (Nov 2022 – Oct 2026)", xlim=(0, per_day.max() * 1.45))
ax.grid(axis="x", alpha=0.25)
plt.tight_layout(); fig.savefig(OUT / "a2_promo_days.png", bbox_inches="tight"); plt.show()

**What this shows.** Every year has 3 Black Friday days and 6 Boxing Week days. Together they bring in **~5% of yearly revenue** (4.8–5.8%) and give away **~$142K a year** in discounts. Sale days are big ($38–43K vs $16.6K on a normal day), but ordinary Nov–Dec days are already $29K. So **most of the holiday peak is full-price gift and winter shopping**, not the sales. The holiday season (Nov–Dec) is 26–30% of each year's revenue.

## 4. Key numbers

In [ ]:
summary = {
    "December vs. average month (store)": f"{store_idx['Dec']:.0f} (≈{store_idx['Dec'] / 100:.1f}×)",
    "Quietest months (store)": ", ".join(f"{m} {v:.0f}" for m, v in store_idx.nsmallest(3).items()),
    "Sep–Oct (back to school / fall)": f"{store_idx['Sep']:.0f} / {store_idx['Oct']:.0f}",
    "Outerwear: Dec peak / Apr–Aug low": f"{cat_idx.loc['Outerwear', 12]:.0f} / {cat_idx.loc['Outerwear', [4,5,6,7,8]].min():.0f}",
    "Outerwear share of its year in Nov–Jan": f"{peaks.loc['Outerwear', 'Nov–Jan share of year']:.0%}",
    "Shorts / Tank Tops peak": f"{peaks.loc['Shorts', 'peak month']} {peaks.loc['Shorts', 'peak index']:.0f}, "
                               f"Nov–Feb ≈ {cat_idx.loc['Shorts', [11,12,1,2]].mean():.0f}",
    "Promo days per year (BF + BW)": f"{promo['BF days'].mean():.0f} + {promo['BW days'].mean():.0f}",
    "Promo share of annual revenue": f"{promo['promo share of year'].mean():.1%}",
    "Discount given per year (avg)": f"${promo['discount given'].mean():,.0f}",
    "Sale day vs. other Nov–Dec day": f"{promo['promo $/day ÷ other Nov–Dec $/day'].mean():.2f}×",
    "Seasonal profile stable pre-competitor?": f"correlation {store_idx.corr(pre_idx):.2f}",
}
pd.Series(summary, name="value").to_frame()

## Conclusions and caveats
1. **Seasonality is large and stable**: December is ~1.9× an average month, February/June/July ~0.76×. Four calculation methods agree.
2. **Winter and summer products swing in opposite directions**; outerwear sells half its year in Nov–Jan.
3. **The holiday sales are small** (~5% of revenue, 9 days); the December peak is mostly full-price shopping.
4. **How to compare periods fairly:** always compare **the same months year over year**. Loyalty launched in June (index 76) and repairs in April (82), both quiet months. A raw before/after comparison around either launch would be skewed by seasonality.

**Caveats.** Four years gives only four observations of each month. Sale days may pull purchases forward from nearby days, which we don't measure.

---
*Team reference: plan item A-2 (guiding question 2 in the case brief). Findings F6–F8 in `results/a_competitor/findings.md`. Charts saved to `results/a_competitor/`.*